# 3_1 规则事实补全

为 new 数据缺少 X 的五个 ENV 建立候选来源与人工审核模板。此页**不会自动修改**正式 `rule_facts.csv` 或 `environment_metadata.csv`。

In [4]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name != 'python':
    ROOT = ROOT / 'python'
PROJECT = ROOT.parent
DATA = PROJECT / '数据'
X_DATA = DATA / '02_X匹配数据'
WORK = DATA / '06_1016工作数据' / '规则事实补全'
WORK.mkdir(parents=True, exist_ok=True)

TARGET_ENVS = [
    '抖音|双11|2021', '抖音|双11|2022', '抖音|双11|2023',
    '拼多多|双11|2021', '拼多多|双11|2022',
]
CATEGORIES = [f'C{i}' for i in range(1, 6)] + [f'R{i}' for i in range(1, 7)]
TARGET_ENVS

['抖音|双11|2021', '抖音|双11|2022', '抖音|双11|2023', '拼多多|双11|2021', '拼多多|双11|2022']

In [6]:
candidate_sources = pd.DataFrame([
    ['抖音|双11|2021', 'DY21_D11_S1', '招商规则转载；待核对原规则页或存档', 'https://www.sohu.com/a/492997242_284687'],
    ['抖音|双11|2022', 'DY22_D11_S1', '招商规则转载；待核对抖店学习中心原页', 'https://www.roundchic.com/h-nd-353.html'],
    ['抖音|双11|2023', 'DY23_D11_S1', '活动规则与玩法报道；待核对原规则页或存档', 'https://www.cnecn.com.cn/hangye/lingshou/2023-09-19/13732.html'],
    ['拼多多|双11|2021', 'PDD21_D11_S1', '万券齐发规则转载；待核对平台活动页存档', 'https://m.maijiaw.com/article/499606'],
    ['拼多多|双11|2022', 'PDD22_D11_S1', '活动期、百亿补贴、跨店满减报道；待补原规则页', 'https://www.itheat.com/view/34081.html'],
], columns=['ENV', 'candidate_source_id', 'source_note', 'source_url'])
candidate_path = WORK / '候选来源.csv'
candidate_sources.to_csv(candidate_path, index=False, encoding='utf-8-sig')
candidate_sources

,ENV,candidate_source_id,source_note,source_url
0,抖音|双11|2021,DY21_D11_S1,招商规则转载；待核对原规则页或存档,https://www.sohu.com/a/492997242_284687
1,抖音|双11|2022,DY22_D11_S1,招商规则转载；待核对抖店学习中心原页,https://www.roundchic.com/h-nd-353.html
2,抖音|双11|2023,DY23_D11_S1,活动规则与玩法报道；待核对原规则页或存档,https://www.cnecn.com.cn/hangye/lingshou/2023-...
3,拼多多|双11|2021,PDD21_D11_S1,万券齐发规则转载；待核对平台活动页存档,https://m.maijiaw.com/article/499606
4,拼多多|双11|2022,PDD22_D11_S1,活动期、百亿补贴、跨店满减报道；待补原规则页,https://www.itheat.com/view/34081.html


In [8]:
definitions = pd.read_csv(X_DATA / 'category_definitions.csv', dtype=str).set_index('category')
rows = []
for env in TARGET_ENVS:
    for category in CATEGORIES:
        rows.append({
            'ENV': env,
            'category': category,
            'category_name': definitions.loc[category, 'category_name'],
            'review_state': 'unknown',  # 仅可填 observed_yes / explicit_no / unknown
            'source_id': '',
            'source_url': '',
            'source_locator': '',
            'fact_paraphrase': '',
            'coding_reason': '',
            'reviewer_note': '',
        })
review = pd.DataFrame(rows)
review_path = WORK / '人工审核_规则状态.csv'
if not review_path.exists():
    review.to_csv(review_path, index=False, encoding='utf-8-sig')
    print('已创建：', review_path)
else:
    print('保留已有人工审核表：', review_path)
review

已创建： /Users/xuxiaoxue/Documents/Writings/DA-holiday-work/reasons-analysis-of-shopping-and-rules/lasso-1016/数据/06_1016工作数据/规则事实补全/人工审核_规则状态.csv


,ENV,category,category_name,review_state,source_id,source_url,source_locator,fact_paraphrase,coding_reason,reviewer_note
0,抖音|双11|2021,C1,准入数量或金额门槛,unknown,,,,,,
1,抖音|双11|2021,C2,商品或店铺适用范围,unknown,,,,,,
2,抖音|双11|2021,C3,身份或指定渠道,unknown,,,,,,
3,抖音|双11|2021,C4,主动领取或额外任务,unknown,,,,,,
4,抖音|双11|2021,C5,领取或参与次数配额,unknown,,,,,,
5,抖音|双11|2021,R1,合并或累计,unknown,,,,,,
6,抖音|双11|2021,R2,优惠叠加,unknown,,,,,,
7,抖音|双11|2021,R3,互斥或择一,unknown,,,,,,
8,抖音|双11|2021,R4,计算基数改变或非面额换算,unknown,,,,,,
9,抖音|双11|2021,R5,优惠金额封顶或封底,unknown,,,,,,


## 填表规则

- `observed_yes`：来源直接支持该类别，必须填来源、定位、释义与编码理由。
- `explicit_no`：来源明确说该类规则不存在；不能因为没看到就填否。
- `unknown`：证据不足，保持未知。
- 不可用评论文本或“预计有此玩法”填规则事实。
- 完成后运行下一格，只做完整性检查，不写正式规则库。

In [11]:
reviewed = pd.read_csv(WORK / '人工审核_规则状态.csv', dtype=str, keep_default_na=False)
required = {'ENV', 'category', 'review_state', 'source_id', 'source_url', 'source_locator', 'fact_paraphrase', 'coding_reason'}
missing = required - set(reviewed.columns)
if missing:
    raise ValueError(f'审核表缺列：{sorted(missing)}')
if set(reviewed.review_state) - {'observed_yes', 'explicit_no', 'unknown'}:
    raise ValueError('review_state 只能是 observed_yes / explicit_no / unknown')
if reviewed.duplicated(['ENV', 'category']).any():
    raise ValueError('同一 ENV 与类别不可重复')
for env in TARGET_ENVS:
    states = reviewed.loc[reviewed.ENV.eq(env), 'category'].tolist()
    if sorted(states) != sorted(CATEGORIES):
        raise ValueError(f'{env} 缺少或多出类别')
yes = reviewed.review_state.eq('observed_yes')
needed = ['source_id', 'source_url', 'source_locator', 'fact_paraphrase', 'coding_reason']
if (reviewed.loc[yes, needed] == '').any(axis=None):
    raise ValueError('observed_yes 必须填全来源、定位、释义、编码理由')

summary = reviewed.pivot_table(index='ENV', columns='review_state', values='category', aggfunc='count', fill_value=0)
summary

review_state,unknown
ENV,
抖音|双11|2021,11
抖音|双11|2022,11
抖音|双11|2023,11
拼多多|双11|2021,11
拼多多|双11|2022,11


审核表通过后发给我。我会再核对 D1–D4 覆盖、来源准入与语义事实去重；只有达标的 ENV 才写入正式规则库并获得 X。